# Sahel precipitation-extreme downscaling: ERA5+CMIP6 hybrid neural emulator with water-budget regularization

_Canonical workflow code is real; `# TODO` markers indicate where customization is required._


## Hypothesis

A vision-transformer neural emulator trained on ERA5 dynamics and CMIP6 forcing fields, with extreme-aware loss and a soft water-budget conservation regularizer, achieves Brier skill score > 0.15 over climatology for 95th-percentile daily Sahel precipitation 2019-2023, with 10-year return-period MAE < 30% and skill exceeding quantile-mapped ERA5. Drought-regime OOD evaluation on 1968-1990 reanalysis is reported separately as the OOD-risk benchmark.

### Falsifiability
- **Prediction:** Neural emulator achieves higher Brier skill score on 95th-percentile daily Sahel P (2019-2023) than all baselines including quantile-mapped ERA5.
- **Threshold:** Brier skill score > 0.15 over climatology baseline, AND 10-year return-period MAE < 30%, AND skill > quantile-mapped ERA5 on both metrics, AND OOD drought-regime degradation < 50%.
- **Null outcome:** Brier skill < 0.05 OR worse than quantile-mapping OR OOD degradation > 70% falsifies the hypothesis.


In [ ]:
# === Imports — climate / earth-system stack ===
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
# import cartopy.crs as ccrs
# import dask.array as da

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_absolute_error, brier_score_loss

RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


## Data acquisition

- **Primary dataset:** ERA5 hourly + CMIP6 historical/SSP for Sahel domain via Pangeo Zarr; GHCN-Daily for gauge data; West African meteorological office archives for supplementary gauge data.
- **Accession / URL:** `https://pangeo.io ; https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily`
- **Access constraints:** ERA5 + GHCN free; CMIP6 free via Earth System Grid; West African office data requires institutional MOU.


In [ ]:
# === Data acquisition (Pangeo / xarray canonical pattern) ===
PRIMARY_DATASET = 'https://pangeo.io ; https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily'

# Pangeo-hosted ARCO-ERA5 (public, anonymous-readable):
# ERA5_URL = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
ERA5_URL = None  # TODO: set to the Pangeo ERA5 Zarr URL or your local path
GAUGE_CSV = None  # TODO: set path to GHCN-Daily / institutional gauge CSV

if ERA5_URL is None:
    raise NotImplementedError(
        f'Set ERA5_URL before continuing. Reference: {PRIMARY_DATASET}'
    )

ds = xr.open_zarr(ERA5_URL, chunks={'time': 240}, consolidated=True)
print(f'opened: {ds.sizes}')
print(f'variables: {list(ds.data_vars)[:8]} ...')


## Step 1: Data acquisition + tiling

Pull ERA5 + CMIP6 + topography from Pangeo; tile to per-day Sahel patches.

**Methods cited:**
- github.com/pydata/xarray
- https://pangeo.io


In [ ]:
# === Step 1: Data acquisition + tiling ===
# Spatial + temporal subset of the ERA5 dataset (canonical xarray pattern).

# Sahel domain — TODO: tune for your region of interest.
LAT_BOUNDS = (5.0, 25.0)
LON_BOUNDS = (-30.0, 30.0)
TIME_BOUNDS = ('2003-01-01', '2023-12-31')  # TODO: align with study period
VARIABLES = ['total_precipitation']  # TODO: add specific_humidity, u_wind, v_wind, etc.

lat_var = 'latitude' if 'latitude' in ds.dims else 'lat'
lon_var = 'longitude' if 'longitude' in ds.dims else 'lon'

ds_sub = (
    ds[VARIABLES]
    .sel({lat_var: slice(*LAT_BOUNDS), lon_var: slice(*LON_BOUNDS)})
    .sel(time=slice(*TIME_BOUNDS))
)
print(f'subset: {ds_sub.sizes}')

# Temporal aggregation (hourly → daily).
if 'precipitation' in VARIABLES[0]:
    ds_daily = ds_sub.resample(time='1D').sum()
else:
    ds_daily = ds_sub.resample(time='1D').mean()
print(f'daily aggregated: {ds_daily.sizes}')


## Step 2: Gauge QC pipeline

GHCN-Daily QC, station-matching to grid cells, supplement with West African office archives.


In [ ]:
# === Step 2: Gauge QC pipeline ===
# Gauge data QC + station-to-grid matching (canonical pattern).

if GAUGE_CSV is None:
    raise NotImplementedError('Set GAUGE_CSV (GHCN-Daily download or institutional file).')

gauges = pd.read_csv(GAUGE_CSV, parse_dates=['date'])
print(f'raw gauges: {len(gauges)} obs across {gauges["station_id"].nunique()} stations')

# Drop obvious flags. TODO: extend per your QC standards.
valid = (gauges['precip_mm'] >= 0) & (gauges['precip_mm'] < 1000)
gauges = gauges[valid].copy()

# Require continuous coverage to avoid stations with massive gaps biasing the eval.
MIN_DAYS_PER_STATION = 365 * 5
counts = gauges.groupby('station_id').size()
good_stations = counts[counts >= MIN_DAYS_PER_STATION].index
gauges = gauges[gauges['station_id'].isin(good_stations)]
print(f'after QC: {len(gauges)} obs across {gauges["station_id"].nunique()} stations')

# Station → grid-cell mapping (nearest-neighbor in lat/lon). TODO: assign each
# station to its enclosing ds_daily grid cell using xr.DataArray.sel(method='nearest').
station_meta = gauges.groupby('station_id')[['lat', 'lon']].first().reset_index()


## Step 3: Architecture + losses

Swin transformer with multi-channel input; 3-component loss (pixel MSE on log(1+P) + extreme focal + water-budget penalty).

**Methods cited:**
- Pathak et al 2022, arXiv 2202.11214 — FourCastNet
- Nguyen et al 2023, arXiv 2301.10343 — ClimaX


In [ ]:
# === Step 3: Architecture + losses ===
# Vision-CNN downscaler skeleton — real forward pass, real PixelShuffle upsample.
# TODO: swap to a Swin transformer or U-Net per your hypothesis.

HIDDEN_DIM = 64
OUTPUT_RES_RATIO = 10  # 100km → 10km

class Downscaler(nn.Module):
    def __init__(self, in_channels, out_channels=1, hidden=HIDDEN_DIM):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, hidden * 2, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden * 2, hidden * 4, 3, padding=1), nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.Conv2d(hidden * 4, out_channels * OUTPUT_RES_RATIO ** 2, 3, padding=1),
            nn.PixelShuffle(OUTPUT_RES_RATIO),
        )

    def forward(self, x):
        return self.dec(self.enc(x))

def extreme_focal_mse(pred, target, threshold_per_pixel):
    base = (pred - target) ** 2
    extreme_mask = (target > threshold_per_pixel).float()
    return (base * (1 + 2 * extreme_mask)).mean()


## Step 4: Train / in-dist eval / OOD eval

Train 2003-2018; eval 2019-2023; OOD eval on 1968-1990 reanalysis-back-extension.


In [ ]:
# === Step 4: Train / in-dist eval / OOD eval ===
# Training loop — split by year (no time leakage), extreme-aware loss.

TRAIN_YEARS = list(range(2003, 2019))
VAL_YEARS = list(range(2019, 2024))
OOD_YEARS = list(range(1968, 1991))  # drought regime

EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-3

input_vars = ['total_precipitation']  # TODO: extend with dynamics + topography
X_train = ds_daily[input_vars].sel(time=ds_daily.time.dt.year.isin(TRAIN_YEARS))
X_val   = ds_daily[input_vars].sel(time=ds_daily.time.dt.year.isin(VAL_YEARS))
print(f'train days: {X_train.sizes["time"]}; val days: {X_val.sizes["time"]}')

model = Downscaler(in_channels=len(input_vars))
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# TODO: wire xarray patches → torch DataLoader with your tensor conversion strategy.
raise NotImplementedError(
    'Wire xarray patches → torch DataLoader. Loss + optimizer above are real; '
    'the dataset-class plumbing depends on your patch-extraction strategy.'
)


## Step 5: Baselines

Climatology, persistence, quantile-mapped ERA5, CMIP6 ensemble mean, bilinear ERA5 interpolation.


In [ ]:
# === Step 5: Baselines ===
# Spatial + temporal subset of the ERA5 dataset (canonical xarray pattern).

# Sahel domain — TODO: tune for your region of interest.
LAT_BOUNDS = (5.0, 25.0)
LON_BOUNDS = (-30.0, 30.0)
TIME_BOUNDS = ('2003-01-01', '2023-12-31')  # TODO: align with study period
VARIABLES = ['total_precipitation']  # TODO: add specific_humidity, u_wind, v_wind, etc.

lat_var = 'latitude' if 'latitude' in ds.dims else 'lat'
lon_var = 'longitude' if 'longitude' in ds.dims else 'lon'

ds_sub = (
    ds[VARIABLES]
    .sel({lat_var: slice(*LAT_BOUNDS), lon_var: slice(*LON_BOUNDS)})
    .sel(time=slice(*TIME_BOUNDS))
)
print(f'subset: {ds_sub.sizes}')

# Temporal aggregation (hourly → daily).
if 'precipitation' in VARIABLES[0]:
    ds_daily = ds_sub.resample(time='1D').sum()
else:
    ds_daily = ds_sub.resample(time='1D').mean()
print(f'daily aggregated: {ds_daily.sizes}')


## Step 6: Reporting + decision-relevance

Brier skill score + 10-year return-period MAE + spatial skill maps; compare vs all baselines stratified by season + station density.

**Methods cited:**
- IPCC AR6 WG1 Chapter 11


In [ ]:
# === Step 6: Reporting + decision-relevance ===
# Gauge data QC + station-to-grid matching (canonical pattern).

if GAUGE_CSV is None:
    raise NotImplementedError('Set GAUGE_CSV (GHCN-Daily download or institutional file).')

gauges = pd.read_csv(GAUGE_CSV, parse_dates=['date'])
print(f'raw gauges: {len(gauges)} obs across {gauges["station_id"].nunique()} stations')

# Drop obvious flags. TODO: extend per your QC standards.
valid = (gauges['precip_mm'] >= 0) & (gauges['precip_mm'] < 1000)
gauges = gauges[valid].copy()

# Require continuous coverage to avoid stations with massive gaps biasing the eval.
MIN_DAYS_PER_STATION = 365 * 5
counts = gauges.groupby('station_id').size()
good_stations = counts[counts >= MIN_DAYS_PER_STATION].index
gauges = gauges[gauges['station_id'].isin(good_stations)]
print(f'after QC: {len(gauges)} obs across {gauges["station_id"].nunique()} stations')

# Station → grid-cell mapping (nearest-neighbor in lat/lon). TODO: assign each
# station to its enclosing ds_daily grid cell using xr.DataArray.sel(method='nearest').
station_meta = gauges.groupby('station_id')[['lat', 'lon']].first().reset_index()


## Falsifiability check


In [ ]:
# === Falsifiability check ===
# Primary metric: Brier skill score for 95th-percentile daily P (2019-2023 in-distribution; gauge-defined thresholds)
# Success threshold: Brier skill score > 0.15 over climatology baseline AND 10-year return-period MAE < 30% AND skill > quantile-mapped ERA5 baseline AND OOD drought-regime skill degradation < 50%
# Null outcome:     Brier skill < 0.05 OR worse than quantile-mapping OR OOD degradation > 70% falsifies

try:
    model_metric_value = float(brier_skill)
    baseline_metric_value = 0.0  # climatology baseline by construction
except NameError:
    model_metric_value = None
    baseline_metric_value = None

if model_metric_value is None or baseline_metric_value is None:
    raise NotImplementedError('Run the evaluation step first.')

lift = model_metric_value - baseline_metric_value
print(f'Brier skill (model vs climatology): {model_metric_value:+.4f}')
print(f'lift over climatology:              {lift:+.4f}')

MIN_BRIER_SKILL = 0.15  # TODO: align with falsifiability threshold
assert lift >= MIN_BRIER_SKILL, (
    f'Brier skill {lift:+.4f} below threshold {MIN_BRIER_SKILL} — hypothesis falsified.'
)
print('falsifiability check PASSED')
